# A-S-FLC: Asymmetric Signed Force-Loop-Chain

**Turn any LLM into a force-guided decision navigator.**

This notebook lets you try all 3 modes of A-S-FLC:
1. **Single-shot** — fast, one LLM call
2. **What-If** — adds worst-case stress testing (recommended for high-stakes)
3. **Hybrid** — LLM generates tree, Python engine scores deterministically

No install needed. Just add your free Groq API key and run.

GitHub: [denial-web/a-s-flc-llm-enhancer](https://github.com/denial-web/a-s-flc-llm-enhancer)

## Setup

Get a free API key at [console.groq.com](https://console.groq.com) (takes 30 seconds).

In [ ]:
!git clone https://github.com/denial-web/a-s-flc-llm-enhancer.git
%cd a-s-flc-llm-enhancer
!pip install -q pydantic groq

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

## Mode 1: Single-Shot (Fast)

The LLM analyses, scores, and ranks all chains in one call using the FG-CoT prompt.
Best for quick decisions where speed matters.

In [ ]:
from config import A_S_FLC_Config
from inference.wrapper import A_S_FLC_Wrapper

config = A_S_FLC_Config()
wrapper = A_S_FLC_Wrapper(config)

query = "Should I invest $10k in index funds or crypto?"
print(f"Query: {query}\n")

result = wrapper.decide(query)
print(result.model_dump_json(indent=2))

### Reading the output

- **chosen_action**: the recommended decision
- **breakdown.positives**: exact known benefits (0-10)
- **breakdown.negatives_buffered**: estimated costs + conservative buffer
- **breakdown.net**: positives minus buffered negatives
- **all_chains**: every alternative scored and ranked
- **stability_score**: how stable the score is across LCDI loop iterations (1.0 = perfectly stable)

## Mode 2: What-If Stress Testing (Recommended for High-Stakes)

Same as single-shot, but the LLM also runs a worst-case scenario on the top chains:
*"What if the largest negative event occurs at full severity?"*

Chains where the net drops >15% get flagged as high-risk.

In [ ]:
query = "Should I quit my stable job to pursue a 3-month coding bootcamp?"
print(f"Query: {query}\n")

result = wrapper.decide_whatif(query)
print(result.model_dump_json(indent=2))

print("\n--- Risk Analysis ---")
print(f"What-If: {result.what_if_summary}")
print(f"Risk flags: {result.risk_flags if result.risk_flags else 'None — all chains stable'}")

## Mode 3: Hybrid (Deterministic Scoring)

The LLM only generates the event tree. All scoring, buffering, and ranking
is done by the deterministic Python engine — same tree always produces the same score.

Best when you need auditability and reproducibility.

In [ ]:
query = "Plan my trip from Singapore to Tokyo on a $1200 budget with max comfort"
print(f"Query: {query}\n")

result = wrapper.decide_hybrid(query)
print(result.model_dump_json(indent=2))

## Try Your Own Query

Change the query below to any decision you're facing. Pick the mode that fits:
- `wrapper.decide(query)` — fast
- `wrapper.decide_whatif(query)` — risk-aware
- `wrapper.decide_hybrid(query)` — deterministic

In [ ]:
query = "Should I buy a house now or keep renting and invest the down payment?"

result = wrapper.decide_whatif(query)

print(f"Decision: {result.chosen_action}")
print(f"Net score: {result.breakdown.net:+.2f}")
print(f"Stability: {result.stability_score}")
print(f"Chains compared: {len(result.all_chains)}")
if result.what_if_summary:
    print(f"\nWhat-If: {result.what_if_summary}")
if result.risk_flags:
    print(f"Risk flags: {result.risk_flags}")

print("\n--- All Chains ---")
for chain in result.all_chains:
    flag = " ⚠️ HIGH-RISK" if chain.chain_id in (result.risk_flags or []) else ""
    print(f"  {chain.chain_id}: net={chain.net:+.2f} (pos={chain.positives}, neg_buf={chain.negatives_buffered}){flag}")

## The Core Insight: Asymmetric Signing

Standard CoT treats pros and cons equally. A-S-FLC doesn't:
- **Positives** = exact, 100% trusted
- **Negatives** = estimated + buffer proportional to uncertainty

This catches "trap" decisions where uncertain downsides are underestimated.

Run the edge-case proof below — no LLM needed:

In [ ]:
from core.types import EventNode
from core.chains import enumerate_paths
from core.loops import run_all_chains
from core.forces import rank_chains

trap = EventNode(id="trap", description="Trap: looks great on paper",
                 positives=9.5, negatives=1.0, transition_prob=1.0,
                 children=[EventNode(id="trap-hidden", description="Hidden massive downside",
                                     positives=0.0, negatives=8.0, transition_prob=0.8)])

honest = EventNode(id="honest", description="Honest: moderate but reliable",
                   positives=5.0, negatives=1.5, transition_prob=1.0,
                   children=[EventNode(id="honest-stable", description="Stable outcome",
                                       positives=2.0, negatives=1.0, transition_prob=0.95)])

root = EventNode(id="root", description="Decision", children=[trap, honest])
paths = enumerate_paths(root, config)

raw_scores = rank_chains(paths, config)
loop_results = run_all_chains(paths, config)

print("=== Adversarial Flip Demo ===")
print(f"\nWithout A-S-FLC buffering, the 'trap' looks better.")
print(f"With buffering, the honest option wins:\n")
for bd, history, stability in loop_results:
    label = "TRAP" if "trap" in bd.events[0].lower() else "HONEST"
    print(f"  {label}: net={bd.net:+.2f}  (pos={bd.positives:.2f}, neg_buf={bd.negatives_buffered:.2f})")

winner = loop_results[0][0]
print(f"\nWinner: {'HONEST' if 'honest' in winner.events[0].lower() else 'TRAP'} option")
print(f"The buffer caught the trap.")

---

**Like this?** Star the repo: [github.com/denial-web/a-s-flc-llm-enhancer](https://github.com/denial-web/a-s-flc-llm-enhancer)

**Questions?** Open an issue on GitHub.